In [1]:
# Movie Recommendation System

In [ ]:
import numpy as np
import pandas as pd
import regex as re
from gensim.models import Word2Vec

import gensim.downloader as gen_download

from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Lets load the dataset
data = pd.read_csv(r'/content/drive/MyDrive/GenAI Week1 (NLP)/DAY 4/yaPJKg.csv')

# Change the path of the data for your use

In [ ]:
data.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
# lets see the shape of the data

data.shape

(9742, 3)

In [ ]:
# Lets check for the missing values

data.isnull().sum()

,0
movieId,0
title,0
genres,0


In [ ]:
# There is no missing values in the data

In [ ]:
#  Lets convert text to tokens

data['genres_tokens'] = data['genres'].str.lower().str.split(pat='|')

In [ ]:
data

,movieId,title,genres,genres_tokens
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,"[adventure, animation, children, comedy, fantasy]"
1,2,Jumanji (1995),Adventure|Children|Fantasy,"[adventure, children, fantasy]"
2,3,Grumpier Old Men (1995),Comedy|Romance,"[comedy, romance]"
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,"[comedy, drama, romance]"
4,5,Father of the Bride Part II (1995),Comedy,[comedy]
...,...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy,"[action, animation, comedy, fantasy]"
9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy,"[animation, comedy, fantasy]"
9739,193585,Flint (2017),Drama,[drama]
9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation,"[action, animation]"


### SKIPGRAM MODDEL TRAINING

In [ ]:
# Lets train the SkipGram model on our data and generate word vectors for tokens

embed_model = Word2Vec(sentences=data['genres_tokens'],window=1,vector_size=30,sg=1)

In [ ]:
embed_model.wv['children']

array([-0.16756962, -0.04561825,  0.4204095 , -0.21484911,  0.165316  ,
       -0.0039331 ,  0.30791295,  0.27414274, -0.32902113, -0.10341905,
        0.1934008 ,  0.0458397 , -0.10424507, -0.2936167 ,  0.2563251 ,
       -0.1930611 ,  0.51844084,  0.07529254, -0.3014852 ,  0.17716731,
        0.0644674 , -0.11941597,  0.08647055,  0.26981068, -0.05224409,
        0.2236652 ,  0.47149068,  0.37919083, -0.07739005, -0.19374084],
      dtype=float32)

In [ ]:
embed_model.wv.most_similar('horror',topn=5)

[('thriller', 0.9931560754776001),
 ('sci-fi', 0.9924430847167969),
 ('action', 0.9923739433288574),
 ('drama', 0.9921454787254333),
 ('crime', 0.9918853044509888)]

In [ ]:
# Vocabulary

embed_model.wv.key_to_index

{'drama': 0,
 'comedy': 1,
 'thriller': 2,
 'action': 3,
 'romance': 4,
 'adventure': 5,
 'crime': 6,
 'sci-fi': 7,
 'horror': 8,
 'fantasy': 9,
 'children': 10,
 'animation': 11,
 'mystery': 12,
 'documentary': 13,
 'war': 14,
 'musical': 15,
 'western': 16,
 'imax': 17,
 'film-noir': 18,
 '(no genres listed)': 19}

In [ ]:
# As we can see the token names as no genres listed. Better remove movies where genre is not provided

drop_indexes = data[data['genres']=='(no genres listed)'].index
data.drop(index=drop_indexes,inplace=True)
data.reset_index(inplace=True)

In [ ]:
embed_model = Word2Vec(sentences=data['genres_tokens'],window=1,vector_size=20,sg=1)

In [ ]:
embed_model.wv['children']

array([-0.46748608,  0.29363307,  0.05260868, -0.07573728,  0.3592818 ,
       -0.00846707,  0.18546908,  0.50605583, -0.35745877,  0.16151386,
        0.21948183, -0.27923802, -0.02948477, -0.2286275 ,  0.23929825,
       -0.04027289,  0.6461364 ,  0.11704704, -0.5028493 , -0.08130468],
      dtype=float32)

In [ ]:
embed_model.wv.most_similar('horror',topn=5)

[('animation', 0.9915146231651306),
 ('mystery', 0.9901582598686218),
 ('fantasy', 0.9870805144309998),
 ('children', 0.9865832328796387),
 ('imax', 0.985885500907898)]

## GLOVE TRAINING (PRE-TRAINED)

In [ ]:
# Lets try some pre trained model (Glove)

model_glove = gen_download.load('glove-twitter-25')

In [ ]:
model_glove['documentary']

array([ 0.30368 ,  0.081423,  0.79418 , -0.3667  ,  0.61585 , -0.89648 ,
        1.6116  , -0.74739 ,  0.21872 , -1.4537  ,  0.52975 ,  1.3648  ,
       -2.4574  ,  0.18059 ,  0.35748 ,  0.44751 ,  0.011323,  0.77678 ,
       -0.10042 ,  0.33302 , -0.40845 , -0.24913 , -0.86413 , -1.3932  ,
        0.70769 ], dtype=float32)

Using trainable model as some of the words are not present in the Pre trained model.

In [ ]:
# Lets convert these tokens to word embedding

data['genre_embedding'] = data['genres_tokens'].apply(lambda x:[embed_model.wv[w] for w in x])

In [ ]:
data['genre_embedding'][0]

[array([-0.49919525,  0.28057653,  0.11958571, -0.02145502,  0.34448266,
        -0.09068803,  0.20604604,  0.5160255 , -0.311345  ,  0.08546682,
         0.23464768, -0.30683917, -0.13297948, -0.16956508,  0.19958748,
         0.03654546,  0.6232621 ,  0.1042832 , -0.44791576, -0.04649246],
       dtype=float32),
 array([-0.51496726,  0.24445404,  0.07036521, -0.11472321,  0.36294708,
        -0.09134925,  0.2563428 ,  0.5270517 , -0.34644118,  0.14392701,
         0.2760862 , -0.304022  , -0.14506152, -0.24599238,  0.13545097,
        -0.01272075,  0.60497326,  0.07236737, -0.43727356, -0.0545108 ],
       dtype=float32),
 array([-0.46748608,  0.29363307,  0.05260868, -0.07573728,  0.3592818 ,
        -0.00846707,  0.18546908,  0.50605583, -0.35745877,  0.16151386,
         0.21948183, -0.27923802, -0.02948477, -0.2286275 ,  0.23929825,
        -0.04027289,  0.6461364 ,  0.11704704, -0.5028493 , -0.08130468],
       dtype=float32),
 array([-0.4050658 ,  0.28814253,  0.12864316, -0.05

In [ ]:
data['genre_embedding_avg'] = data['genre_embedding'].apply(lambda x: np.mean(x,axis=0))

In [ ]:
data

,index,movieId,title,genres,genres_tokens,genre_embedding,genre_embedding_avg
0,0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,"[adventure, animation, children, comedy, fantasy]","[[-0.49919525, 0.28057653, 0.11958571, -0.0214...","[-0.4779571, 0.27845794, 0.09675365, -0.071662..."
1,1,2,Jumanji (1995),Adventure|Children|Fantasy,"[adventure, children, fantasy]","[[-0.49919525, 0.28057653, 0.11958571, -0.0214...","[-0.48991743, 0.28656438, 0.09491997, -0.06452..."
2,2,3,Grumpier Old Men (1995),Comedy|Romance,"[comedy, romance]","[[-0.4050658, 0.28814253, 0.12864316, -0.05000...","[-0.3865314, 0.2576067, 0.11415124, -0.0302412..."
3,3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,"[comedy, drama, romance]","[[-0.4050658, 0.28814253, 0.12864316, -0.05000...","[-0.40530872, 0.25643215, 0.11538916, -0.02125..."
4,4,5,Father of the Bride Part II (1995),Comedy,[comedy],"[[-0.4050658, 0.28814253, 0.12864316, -0.05000...","[-0.4050658, 0.28814253, 0.12864316, -0.050002..."
...,...,...,...,...,...,...,...
9703,9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy,"[action, animation, comedy, fantasy]","[[-0.4698298, 0.2598916, 0.06234867, -0.119194...","[-0.47323346, 0.26949295, 0.09348064, -0.09507..."
9704,9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy,"[animation, comedy, fantasy]","[[-0.51496726, 0.24445404, 0.070365205, -0.114...","[-0.474368, 0.2726934, 0.10385796, -0.08704082..."
9705,9739,193585,Flint (2017),Drama,[drama],"[[-0.4428633, 0.2540831, 0.117865, -0.00327138...","[-0.4428633, 0.2540831, 0.117865, -0.003271389..."
9706,9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation,"[action, animation]","[[-0.4698298, 0.2598916, 0.06234867, -0.119194...","[-0.49239853, 0.25217283, 0.06635694, -0.11695..."


In [ ]:
data['genre_embedding_avg'][0]

array([-0.4779571 ,  0.27845794,  0.09675365, -0.07166296,  0.3432193 ,
       -0.06663091,  0.20864344,  0.49751487, -0.32709315,  0.11505407,
        0.23289911, -0.2819787 , -0.08249128, -0.23317246,  0.18048617,
       -0.00748362,  0.6237609 ,  0.06927536, -0.46042705, -0.05843516],
      dtype=float32)

## Recommendation System

In [ ]:
# Create a similarity matrix

simi_matrix = cosine_similarity(data['genre_embedding_avg'].tolist())

In [ ]:
simi_matrix

array([[1.0000001 , 0.99935997, 0.9959215 , ..., 0.99071616, 0.9963366 ,
        0.9946263 ],
       [0.99935997, 1.        , 0.994853  , ..., 0.98947483, 0.9943929 ,
        0.99216545],
       [0.9959215 , 0.994853  , 0.9999999 , ..., 0.98871136, 0.9909905 ,
        0.99613243],
       ...,
       [0.99071616, 0.98947483, 0.98871136, ..., 0.99999994, 0.9892264 ,
        0.9853895 ],
       [0.9963366 , 0.9943929 , 0.9909905 , ..., 0.9892264 , 1.        ,
        0.98852104],
       [0.9946263 , 0.99216545, 0.99613243, ..., 0.9853895 , 0.98852104,
        0.9999999 ]], dtype=float32)

In [ ]:
def recommender(selected_movie,nor=5):
  if selected_movie in data['title'].values:
    idx = data[data['title']==selected_movie].index[0]
    top_n_idx = simi_matrix[idx].argsort()[::-1][1:nor+1]
    for i in top_n_idx:
      print(data['title'][i])

  else:
    print('Movie Not Found')


In [ ]:
recommender('Father of the Bride Part II (1995)')

Bio-Dome (1996)
Jeff Ross Roasts the Border (2017)
Extract (2009)
Sibling Rivalry (1990)
Wildcats (1986)


In [ ]:
recommender('Jumanji (1995)')

NeverEnding Story III, The (1994)
Jumanji (1995)
Pan (2015)
Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
Pete's Dragon (2016)


In [ ]:
recommender('Toy Story (1995)')

Turbo (2013)
Asterix and the Vikings (Astérix et les Vikings) (2006)
Tale of Despereaux, The (2008)
Moana (2016)
Monsters, Inc. (2001)
